# SQL Queries

This is the notebook in which I will be performing SQL queries on my arriving and departing flight .csv file. 

#### First, I have to import the cleaned data and the relevant packages

In [14]:
# import

import pandas as pd
import sqlite3

# load

flights = pd.read_csv("schiphol_flights_cleaned_for_sql.csv")

display(flights.head())
print(flights.shape)

,flight_number,airline_code,airline_name,codeshare_frequency,flight_direction,operational_category,airport_code,route_airport_name,route_country_name,route_continent_name,...,aircraft_code,aircraft_registration,scheduled_time,actual_time,scheduled_hour,time_of_day,scheduled_day,day_type,flight_status,delay_minutes
0,HV5960,HV,Transavia Holland,0,Arrival,Scheduled passenger,LIS,Lisbon Humberto Delgado Airport,Portugal,Europe,...,32Q,PHYHY,2026-04-29 00:00:00+02:00,2026-04-29 00:20:21+02:00,0,Overnight,Wednesday,Weekday,Delayed,20.0
1,HV6706,HV,Transavia Holland,0,Arrival,Scheduled passenger,FUE,Fuerteventura Airport,Spain,Africa,...,73H,PHHBK,2026-04-29 00:05:00+02:00,2026-04-29 00:16:49+02:00,0,Overnight,Wednesday,Weekday,On-time,12.0
2,HV6862,HV,Transavia Holland,0,Arrival,Scheduled passenger,ATH,Athens Eleftherios Venizelos International Air...,Greece,Europe,...,73H,PHHZI,2026-04-29 00:05:00+02:00,2026-04-28 23:38:34+02:00,0,Overnight,Wednesday,Weekday,Early,-26.0
3,HV5134,HV,Transavia Holland,0,Arrival,Scheduled passenger,BCN,Josep Tarradellas Barcelona-El Prat Airport,Spain,Europe,...,73H,PHHSB,2026-04-29 00:15:00+02:00,2026-04-29 00:33:43+02:00,0,Overnight,Wednesday,Weekday,Delayed,19.0
4,HV5666,HV,Transavia Holland,1,Arrival,Scheduled passenger,LPA,Gran Canaria Airport,Spain,Africa,...,32Q,PHYHU,2026-04-29 00:15:00+02:00,2026-04-29 00:03:52+02:00,0,Overnight,Wednesday,Weekday,On-time,-11.0


(42490, 23)


#### Then, I need to create some database tables using this dataframe to faciliate later joins, etc.

In [15]:
# create a connection to a SQLite database

conn = sqlite3.connect("schiphol_project.db")

# create a copy of the main flights dataframe for SQL

flights_sql = flights.copy()

# add a simple flight_id column to use as a primary identifier

flights_sql = flights_sql.reset_index(drop=True)
flights_sql["flight_id"] = flights_sql.index + 1

# create a main fact table with the operational flight-level data

flight_fact = flights_sql[
    [
        "flight_id",
        "flight_number",
        "airline_code",
        "airport_code",
        "aircraft_code",
        "terminal",
        "gate",
        "flight_direction",
        "operational_category",
        "scheduled_time",
        "scheduled_hour",
        "scheduled_day",
        "day_type",
        "time_of_day",
        "flight_status",
        "delay_minutes",
        "codeshare_frequency"
    ]
].copy()

# create a separate delay fact table, includes flight number for ID'ing purposes

delay_fact = flights_sql[
    [
        "flight_id",
        "flight_number",
        "flight_direction",
        "scheduled_time",
        "flight_status",
        "delay_minutes"
    ]
].copy()

# add a binary delay flag for whether the flight was delayed by at least 15 minutes

delay_fact["is_delayed_15"] = delay_fact["delay_minutes"].apply(
    lambda delay: 1 if delay >= 15 else 0
)

# add a more readable delay category

delay_fact["delay_category"] = delay_fact["delay_minutes"].apply(
    lambda delay: "On Time" if delay <= 15
    else "Minor Delay" if delay < 60
    else "Moderate Delay" if delay < 180
    else "Major Delay"
)

# create an airline lookup table

airline_dim = flights_sql[
    [
        "airline_code",
        "airline_name"
    ]
].drop_duplicates().copy()

# create an airport / route lookup table

airport_dim = flights_sql[
    [
        "airport_code",
        "route_airport_name",
        "route_country_name",
        "route_continent_name"
    ]
].drop_duplicates().copy()

# create an aircraft lookup table

aircraft_dim = flights_sql[
    [
        "aircraft_code",
        "aircraft_name"
    ]
].drop_duplicates().copy()

# create a terminal lookup table

terminal_dim = flights_sql[
    [
        "terminal"
    ]
].drop_duplicates().copy()

# simplify with map

terminal_dim["terminal_sort"] = terminal_dim["terminal"].map({
    "Terminal 1": 1,
    "Terminal 2": 2,
    "Terminal 3": 3,
    "Terminal 4": 4,
    "Cargo": 5,
    "Unknown": 6
})

# write all tables to SQLite

flight_fact.to_sql("flight_fact", conn, if_exists="replace", index=False)
delay_fact.to_sql("delay_fact", conn, if_exists="replace", index=False)
airline_dim.to_sql("airline_dim", conn, if_exists="replace", index=False)
airport_dim.to_sql("airport_dim", conn, if_exists="replace", index=False)
aircraft_dim.to_sql("aircraft_dim", conn, if_exists="replace", index=False)
terminal_dim.to_sql("terminal_dim", conn, if_exists="replace", index=False)

6

## Query 1 - dataset size and date range

This query checks the size and time range of the final flight-level SQL table. It counts the total number of rows, counts the number of unique flight IDs, and identifies the earliest and latest scheduled flight times in the dataset.

This works by using COUNT(*) to count all rows, COUNT(DISTINCT flight_id) to confirm that each flight ID is unique, and MIN() and MAX() to find the beginning and end of the scheduled flight period.

This query is useful as a basic data validity check before conducting deeper SQL analysis. It confirms that the SQL table contains the expected number of observations and covers the intended time period, and was generated in/loaded in from the Python EDA file correctly.

In [16]:
# query 1

query_01 = """
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT flight_id) AS unique_flight_ids,
    MIN(scheduled_time) AS first_scheduled_time,
    MAX(scheduled_time) AS last_scheduled_time
FROM flight_fact;
"""

pd.read_sql_query(query_01, conn)

,total_rows,unique_flight_ids,first_scheduled_time,last_scheduled_time
0,42490,42490,2026-04-29 00:00:00+02:00,2026-05-28 23:55:00+02:00


## Query 2 - missing value check for important columns

This query checks for missing values across several important columns used in the analysis, including flight number, airline code, airport code, aircraft code, terminal, and delay minutes.

This works by running a separate missing-value count for each column and combining the results into one table using UNION ALL. For the terminal column, the query treats both true missing values and "Unknown" values as missing or incomplete information.

This query matters because missing data can affect both exploratory analysis and modeling. By documenting where missing values remain, the analysis is more transparent about potential data quality limitations.

In [17]:
# query 2

query_02 = """
SELECT 'flight_number' AS column_name, COUNT(*) AS missing_count
FROM flight_fact
WHERE flight_number IS NULL

UNION ALL

SELECT 'airline_code' AS column_name, COUNT(*) AS missing_count
FROM flight_fact
WHERE airline_code IS NULL

UNION ALL

SELECT 'airport_code' AS column_name, COUNT(*) AS missing_count
FROM flight_fact
WHERE airport_code IS NULL

UNION ALL

SELECT 'aircraft_code' AS column_name, COUNT(*) AS missing_count
FROM flight_fact
WHERE aircraft_code IS NULL

UNION ALL

SELECT 'terminal' AS column_name, COUNT(*) AS missing_count
FROM flight_fact
WHERE terminal IS NULL OR terminal = 'Unknown'

UNION ALL

SELECT 'delay_minutes' AS column_name, COUNT(*) AS missing_count
FROM flight_fact
WHERE delay_minutes IS NULL;
"""

pd.read_sql_query(query_02, conn)

,column_name,missing_count
0,flight_number,0
1,airline_code,4
2,airport_code,0
3,aircraft_code,18
4,terminal,1837
5,delay_minutes,1866


This matches with the Python missing values code, so the data was put into a .csv and imported into this notebook successfully!

## Query 3 - flight activity & delay summary, by flight direction

This query compares arrivals and departures by total flight volume, average delay, and the percentage of flights delayed by at least 15 minutes.

This works by grouping the flight table by flight_direction. For each direction, the query counts the number of flights, calculates average delay minutes, and uses a CASE WHEN statement to calculate the share of flights with delays of 15 minutes or more.

This query is important because one of the main operational questions in the project is whether arrival and departure flights experience different delay patterns. The output helps show whether one side of airport operations is more delay-prone than the other.

In [18]:
# query 3

query_03 = """
SELECT
    flight_direction,
    COUNT(*) AS total_flights,
    ROUND(AVG(delay_minutes), 2) AS average_delay_minutes,
    ROUND(
        100.0 * SUM(CASE WHEN delay_minutes >= 15 THEN 1 ELSE 0 END) / COUNT(delay_minutes),
        2
    ) AS delayed_15_percent
FROM flight_fact
WHERE delay_minutes IS NOT NULL
GROUP BY flight_direction
ORDER BY total_flights DESC;
"""

pd.read_sql_query(query_03, conn)

,flight_direction,total_flights,average_delay_minutes,delayed_15_percent
0,Departure,20314,14.57,32.34
1,Arrival,20310,-3.64,13.04


## Query 4 - delay patterns by day of the week and flight direction

This query shows how flight volume and delay performance vary by day of the week and by flight direction.

This works by grouping the data by both scheduled_day and flight_direction. For each day-direction combination, the query calculates total flights, average delay minutes, and the percentage of flights delayed by at least 15 minutes. The ORDER BY CASE section keeps the days of the week in calendar order rather than alphabetical order.

This query is useful because airport operations teams may need different staffing, gate planning, or delay-management strategies depending on the day of the week.

In [19]:
# query 4

query_04 = """
SELECT
    scheduled_day,
    flight_direction,
    COUNT(*) AS total_flights,
    ROUND(AVG(delay_minutes), 2) AS average_delay_minutes,
    ROUND(
        100.0 * SUM(CASE WHEN delay_minutes >= 15 THEN 1 ELSE 0 END) / COUNT(delay_minutes),
        2
    ) AS delayed_15_percent
FROM flight_fact
WHERE delay_minutes IS NOT NULL
GROUP BY scheduled_day, flight_direction
ORDER BY
    CASE scheduled_day
        WHEN 'Monday' THEN 1
        WHEN 'Tuesday' THEN 2
        WHEN 'Wednesday' THEN 3
        WHEN 'Thursday' THEN 4
        WHEN 'Friday' THEN 5
        WHEN 'Saturday' THEN 6
        WHEN 'Sunday' THEN 7
    END,
    flight_direction;
"""

pd.read_sql_query(query_04, conn)

,scheduled_day,flight_direction,total_flights,average_delay_minutes,delayed_15_percent
0,Monday,Arrival,2738,-2.00,14.06
1,Monday,Departure,2717,16.11,37.36
2,Tuesday,Arrival,2663,-5.97,10.06
3,Tuesday,Departure,2689,12.25,29.01
4,Wednesday,Arrival,3408,-3.93,13.12
5,Wednesday,Departure,3381,14.65,31.32
6,Thursday,Arrival,3415,-2.61,14.20
7,Thursday,Departure,3429,15.50,34.09
8,Friday,Arrival,2729,-1.91,14.62
9,Friday,Departure,2732,14.72,32.28


## Query 5 - delay summary by route continent

This query compares flight activity and delay performance across different route continents.

This works by joining the main flight_fact table to the airport_dim table using airport_code. The query then groups the joined data by route continent and flight direction. For each group, it calculates total flights, unique airlines, average delay minutes, and the percentage of flights delayed by at least 15 minutes.

This query is relevant because delay risk may vary by geographic region. Longer-haul or more operationally complex regions may experience different delay patterns than short-haul European routes.

In [20]:
# query 5

query_05 = """
SELECT
    a.route_continent_name,
    f.flight_direction,
    COUNT(*) AS total_flights,
    COUNT(DISTINCT f.airline_code) AS unique_airlines,
    ROUND(AVG(f.delay_minutes), 2) AS average_delay_minutes,
    ROUND(
        100.0 * SUM(CASE WHEN f.delay_minutes >= 15 THEN 1 ELSE 0 END) / COUNT(f.delay_minutes),
        2
    ) AS delayed_15_percent
FROM flight_fact AS f
LEFT JOIN airport_dim AS a
    ON f.airport_code = a.airport_code
WHERE f.delay_minutes IS NOT NULL
GROUP BY a.route_continent_name, f.flight_direction
ORDER BY total_flights DESC;
"""

pd.read_sql_query(query_05, conn)

,route_continent_name,flight_direction,total_flights,unique_airlines,average_delay_minutes,delayed_15_percent
0,Europe,Departure,15798,53,13.10,30.85
1,Europe,Arrival,15711,51,-3.16,12.39
2,Asia,Departure,1865,40,21.34,39.36
3,Asia,Arrival,1840,42,-2.93,17.83
4,North America,Arrival,1727,20,-8.56,12.57
5,North America,Departure,1703,15,17.91,32.88
6,Africa,Arrival,732,14,1.12,17.08
7,Africa,Departure,709,11,21.10,43.02
8,South America,Arrival,300,8,-16.31,10.33
9,South America,Departure,239,3,16.43,40.59


## Query 6 - airline-level flight volume and performance analysis

This query identifies the busiest airlines in the dataset and compares their delay performance.

This works by joining the main flight_fact table to the airline_dim table using airline_code. The query groups by airline name and calculates total flights, arrivals, departures, average codeshare frequency, average delay, and the percentage of flights delayed by at least 15 minutes. The HAVING clause limits the output to airlines with at least 100 flights, which avoids drawing conclusions from very small sample sizes.

This query is useful because airline-level delay patterns are important for both Schiphol airport operations and airline station managers. It helps identify which airlines account for the most activity and which airlines may have higher delay exposure.

In [21]:
# query 6

query_06 = """
SELECT
    ad.airline_name,
    COUNT(*) AS total_flights,
    SUM(CASE WHEN f.flight_direction = 'Arrival' THEN 1 ELSE 0 END) AS arrivals,
    SUM(CASE WHEN f.flight_direction = 'Departure' THEN 1 ELSE 0 END) AS departures,
    ROUND(AVG(f.codeshare_frequency), 2) AS average_codeshares,
    ROUND(AVG(f.delay_minutes), 2) AS average_delay_minutes,
    ROUND(
        100.0 * SUM(CASE WHEN f.delay_minutes >= 15 THEN 1 ELSE 0 END) / COUNT(f.delay_minutes),
        2
    ) AS delayed_15_percent
FROM flight_fact AS f
LEFT JOIN airline_dim AS ad
    ON f.airline_code = ad.airline_code
WHERE f.delay_minutes IS NOT NULL
GROUP BY ad.airline_name
HAVING COUNT(*) >= 100
ORDER BY total_flights DESC;
"""

pd.read_sql_query(query_06, conn)

,airline_name,total_flights,arrivals,departures,average_codeshares,average_delay_minutes,delayed_15_percent
0,KLM Royal Dutch Airlines,21881,10934,10947,4.12,4.44,20.89
1,Transavia Holland,3192,1599,1593,0.88,15.06,38.97
2,easyJet Europe,1780,890,890,0.00,0.76,16.35
3,Delta Air Lines,956,478,478,2.99,6.18,23.85
4,easyJet,952,476,476,0.00,8.13,28.89
5,Arkefly,813,407,406,0.00,11.66,27.68
6,Vueling Airlines,741,370,371,1.90,-0.49,18.62
7,British Airways,685,342,343,0.00,1.70,16.20
8,Scandinavian Airlines System,536,268,268,0.00,-3.52,10.45
9,Lufthansa,496,248,248,3.19,1.92,17.14


## Query 7 - terminal-level flight volume and performance analysis

This query compares flight activity and delay performance by terminal and flight direction.

This works by joining the flight_fact table to the terminal_dim table using the terminal column. The query then groups by terminal and flight direction, calculating total flights, unique airlines, unique gates, average delay minutes, and 15+ minute delay percentage.

This query matters because terminal-level analysis can help identify where airport activity and delay risk are concentrated. This could be useful for staffing, gate allocation, and broader terminal operations planning.

In [22]:
# query 7

query_07 = """
SELECT
    td.terminal,
    f.flight_direction,
    COUNT(*) AS total_flights,
    COUNT(DISTINCT f.airline_code) AS unique_airlines,
    COUNT(DISTINCT f.gate) AS unique_gates,
    ROUND(AVG(f.delay_minutes), 2) AS average_delay_minutes,
    ROUND(
        100.0 * SUM(CASE WHEN f.delay_minutes >= 15 THEN 1 ELSE 0 END) / COUNT(f.delay_minutes),
        2
    ) AS delayed_15_percent
FROM flight_fact AS f
LEFT JOIN terminal_dim AS td
    ON f.terminal = td.terminal
WHERE f.delay_minutes IS NOT NULL
GROUP BY td.terminal, f.flight_direction
ORDER BY td.terminal_sort, f.flight_direction;
"""

pd.read_sql_query(query_07, conn)

,terminal,flight_direction,total_flights,unique_airlines,unique_gates,average_delay_minutes,delayed_15_percent
0,Terminal 1,Arrival,3770,22,61,-3.14,14.03
1,Terminal 1,Departure,10966,21,105,14.26,33.70
2,Terminal 2,Arrival,8965,5,101,-4.07,10.50
3,Terminal 2,Departure,4407,7,84,12.96,28.43
4,Terminal 3,Arrival,4380,65,115,-5.32,13.40
5,Terminal 3,Departure,4202,53,123,14.20,31.34
6,Terminal 4,Arrival,2439,63,108,-4.41,15.33
7,Cargo,Arrival,679,23,0,6.42,27.10
8,Cargo,Departure,677,23,0,31.37,40.77
9,Unknown,Arrival,77,19,5,53.40,42.86


## Query 8 - ranking airlines by delay percentage

This query ranks airlines by the percentage of their flights delayed by at least 15 minutes.

This works in two steps. First, a common table expression creates an airline-level delay summary with total flights, average delay, and 15+ minute delay percentage. Then, the outer query uses the RANK() window function to rank airlines from highest to lowest delay percentage. The query filters airlines with at least 100 flights to avoid ranking airlines with very small sample sizes.

This query is useful because it identifies which airlines have the highest relative delay exposure, not just which airlines operate the most flights, which has operational significance (i.e. resource allocation implications).

In [23]:
# query 8

query_08 = """
WITH airline_delay_summary AS (
    SELECT
        ad.airline_name,
        COUNT(*) AS total_flights,
        ROUND(AVG(f.delay_minutes), 2) AS average_delay_minutes,
        ROUND(
            100.0 * SUM(CASE WHEN f.delay_minutes >= 15 THEN 1 ELSE 0 END) / COUNT(f.delay_minutes),
            2
        ) AS delayed_15_percent
    FROM flight_fact AS f
    LEFT JOIN airline_dim AS ad
        ON f.airline_code = ad.airline_code
    WHERE f.delay_minutes IS NOT NULL
    GROUP BY ad.airline_name
    HAVING COUNT(*) >= 100
)

SELECT
    airline_name,
    total_flights,
    average_delay_minutes,
    delayed_15_percent,
    RANK() OVER (
        ORDER BY delayed_15_percent DESC
    ) AS delay_rate_rank
FROM airline_delay_summary
ORDER BY delay_rate_rank;
"""

pd.read_sql_query(query_08, conn)

,airline_name,total_flights,average_delay_minutes,delayed_15_percent,delay_rate_rank
0,Singapore Airlines,102,18.83,41.18,1
1,Transavia Holland,3192,15.06,38.97,2
2,Etihad Airways,110,4.87,35.45,3
3,Emirates,233,17.80,35.19,4
4,Corendon Dutch Airlines,302,11.70,35.10,5
5,Tarom,120,12.83,35.00,6
6,American Airlines,102,16.61,30.39,7
7,TAP Portugal,176,7.36,28.98,8
8,easyJet,952,8.13,28.89,9
9,Arkefly,813,11.66,27.68,10


## Query 9 - examining time-of-day delay risk by flight direction

This query ranks each time-of-day period by average delay separately for arrivals and departures.

This works by first creating a summary table grouped by flight_direction and time_of_day. The query then uses the RANK() window function with PARTITION BY flight_direction so that arrival time periods are ranked against other arrival time periods, and departure time periods are ranked against other departure time periods.

This query is helpful because the most delay-prone time of day may differ between arrivals and departures. From an operations perspective, this can help identify when delay-management resources may be most needed.

In [24]:
# query 9

query_09 = """
WITH time_delay_summary AS (
    SELECT
        flight_direction,
        time_of_day,
        COUNT(*) AS total_flights,
        ROUND(AVG(delay_minutes), 2) AS average_delay_minutes,
        ROUND(
            100.0 * SUM(CASE WHEN delay_minutes >= 15 THEN 1 ELSE 0 END) / COUNT(delay_minutes),
            2
        ) AS delayed_15_percent
    FROM flight_fact
    WHERE delay_minutes IS NOT NULL
    GROUP BY flight_direction, time_of_day
)

SELECT
    flight_direction,
    time_of_day,
    total_flights,
    average_delay_minutes,
    delayed_15_percent,
    RANK() OVER (
        PARTITION BY flight_direction
        ORDER BY average_delay_minutes DESC
    ) AS delay_rank_within_direction
FROM time_delay_summary
ORDER BY flight_direction, delay_rank_within_direction;
"""

pd.read_sql_query(query_09, conn)

,flight_direction,time_of_day,total_flights,average_delay_minutes,delayed_15_percent,delay_rank_within_direction
0,Arrival,Overnight,754,4.09,26.92,1
1,Arrival,Evening,5691,-1.73,16.76,2
2,Arrival,Afternoon,6651,-1.90,14.22,3
3,Arrival,Morning,7214,-7.56,7.55,4
4,Departure,Afternoon,8065,17.34,39.45,1
5,Departure,Evening,4864,13.67,32.63,2
6,Departure,Morning,7104,12.25,24.51,3
7,Departure,Overnight,281,9.67,21.00,4


## Query 10 - identifying airlines whose average delay > airport average delay

This query identifies airlines whose average delay is higher than the overall average delay across all flights.

This works by grouping flights by airline and calculating each airline’s average delay. The query then compares each airline’s average delay to a subquery that calculates the overall average delay for the entire flight_fact table. The HAVING clause keeps only airlines with at least 100 flights and an average delay above the airport-wide average.

This query is useful because it benchmarks airline delay performance against the overall airport average. Rather than looking at delay values in isolation, it identifies which airlines are performing worse than the broader dataset.

In [25]:
# query 10

query_10 = """
SELECT
    ad.airline_name,
    COUNT(*) AS total_flights,
    ROUND(AVG(f.delay_minutes), 2) AS airline_average_delay,
    ROUND(
        (
            SELECT AVG(delay_minutes)
            FROM flight_fact
            WHERE delay_minutes IS NOT NULL
        ),
        2
    ) AS overall_average_delay
FROM flight_fact AS f
LEFT JOIN airline_dim AS ad
    ON f.airline_code = ad.airline_code
WHERE f.delay_minutes IS NOT NULL
GROUP BY ad.airline_name
HAVING 
    COUNT(*) >= 100
    AND AVG(f.delay_minutes) > (
        SELECT AVG(delay_minutes)
        FROM flight_fact
        WHERE delay_minutes IS NOT NULL
    )
ORDER BY airline_average_delay DESC;
"""

pd.read_sql_query(query_10, conn)

,airline_name,total_flights,airline_average_delay,overall_average_delay
0,Singapore Airlines,102,18.83,5.47
1,Emirates,233,17.80,5.47
2,American Airlines,102,16.61,5.47
3,Transavia Holland,3192,15.06,5.47
4,Tarom,120,12.83,5.47
5,Corendon Dutch Airlines,302,11.70,5.47
6,Arkefly,813,11.66,5.47
7,Air China,110,10.88,5.47
8,easyJet,952,8.13,5.47
9,TAP Portugal,176,7.36,5.47


## Query 11 - identifying routes with above-average delay rates

This query identifies route airports where the percentage of flights delayed by at least 15 minutes is higher than the overall airport delay rate.

This works by joining flight_fact to airport_dim, grouping by route airport, and calculating each route airport’s delay percentage. The query then compares that route-level delay percentage to a subquery that calculates the overall 15+ minute delay percentage for the full dataset. The HAVING clause keeps only route airports with at least 50 flights and above-average delay rates.

This query is useful because it helps identify specific routes or airport pairs that may be associated with higher delay risk. This adds a more detailed geographic dimension to the analysis.

In [26]:
# query 11

query_11 = """
SELECT
    a.airport_code,
    a.route_airport_name,
    a.route_country_name,
    a.route_continent_name,
    COUNT(*) AS total_flights,
    ROUND(AVG(f.delay_minutes), 2) AS average_delay_minutes,
    ROUND(
        100.0 * SUM(CASE WHEN f.delay_minutes >= 15 THEN 1 ELSE 0 END) / COUNT(f.delay_minutes),
        2
    ) AS delayed_15_percent
FROM flight_fact AS f
LEFT JOIN airport_dim AS a
    ON f.airport_code = a.airport_code
WHERE f.delay_minutes IS NOT NULL
GROUP BY 
    a.airport_code,
    a.route_airport_name,
    a.route_country_name,
    a.route_continent_name
HAVING
    COUNT(*) >= 50
    AND (
        100.0 * SUM(CASE WHEN f.delay_minutes >= 15 THEN 1 ELSE 0 END) / COUNT(f.delay_minutes)
    ) > (
        SELECT
            100.0 * SUM(CASE WHEN delay_minutes >= 15 THEN 1 ELSE 0 END) / COUNT(delay_minutes)
        FROM flight_fact
        WHERE delay_minutes IS NOT NULL
    )
ORDER BY delayed_15_percent DESC;
"""

pd.read_sql_query(query_11, conn)

,airport_code,route_airport_name,route_country_name,route_continent_name,total_flights,average_delay_minutes,delayed_15_percent
0,PSA,Pisa International Airport,Italy,Europe,78,25.05,60.26
1,JED,King Abdulaziz International Airport,Saudi Arabia,Asia,62,44.13,59.68
2,CAI,Cairo International Airport,Egypt,Africa,74,18.00,54.05
3,TLV,Ben Gurion International Airport,Israel,Asia,114,18.54,44.74
4,DWC,Al Maktoum International Airport,United Arab Emirates,Asia,70,36.46,44.29
...,...,...,...,...,...,...,...
78,SOF,Sofia Airport,Bulgaria,Europe,60,2.63,23.33
79,KEF,Keflavik International Airport,Iceland,Europe,163,6.69,23.31
80,LGW,London Gatwick Airport,United Kingdom,Europe,289,5.06,23.18
81,YVR,Vancouver International Airport,Canada,North America,52,-4.83,23.08


## Query 12 - delay performance by aircraft type usage

This query identifies the most common aircraft types in the dataset and compares their delay performance.

This works by joining flight_fact to aircraft_dim using aircraft_code. The query groups by aircraft name and calculates total flights, unique airlines, average delay minutes, and the percentage of flights delayed by at least 15 minutes. The HAVING clause keeps only aircraft types with at least 100 flights.

This query matters because aircraft type can reflect route length, airline strategy, and operational complexity. Comparing delay performance across aircraft types adds another operational layer to the analysis.

In [27]:
# query 12

query_12 = """
SELECT
    ac.aircraft_name,
    COUNT(*) AS total_flights,
    COUNT(DISTINCT f.airline_code) AS unique_airlines,
    ROUND(AVG(f.delay_minutes), 2) AS average_delay_minutes,
    ROUND(
        100.0 * SUM(CASE WHEN f.delay_minutes >= 15 THEN 1 ELSE 0 END) / COUNT(f.delay_minutes),
        2
    ) AS delayed_15_percent
FROM flight_fact AS f
LEFT JOIN aircraft_dim AS ac
    ON f.aircraft_code = ac.aircraft_code
WHERE f.delay_minutes IS NOT NULL
GROUP BY ac.aircraft_name
HAVING COUNT(*) >= 100
ORDER BY total_flights DESC;
"""

pd.read_sql_query(query_12, conn)

,aircraft_name,total_flights,unique_airlines,average_delay_minutes,delayed_15_percent
0,Boeing 737-800 (winglets),6513,19,7.97,27.22
1,Airbus A321neo,4665,22,10.01,32.52
2,Embraer 190,4545,5,3.78,17.56
3,Embraer 175 (long wing),3457,2,1.20,13.42
4,Embraer E195-E2,3032,4,7.51,23.15
5,Airbus A320,2316,24,3.65,20.60
6,Airbus A320neo,2283,19,1.36,17.52
7,Airbus A319,1423,11,0.48,17.01
8,Airbus A220-300,1108,8,2.50,16.25
9,Boeing 737-700 (winglets),1091,3,4.80,21.17
